# Lecture 20: Six joints

**The challenge.** Lecture 19 solved a two-link arm exactly. Real arms have six
joints. The state goes from 6 numbers to 18, the limits from 4 to 12, and the number
of ways to ride them at once from 8 to 728. None of the theory changes, so the
question is purely practical: what breaks first when the same code meets a bigger
problem?

The way to answer that is to measure, so this lecture starts with the measurements.
The fixes they led to are two ideas: keep a robot's dynamics as a **program** rather
than a formula, and solve for the riding controls **at run time** rather than in closed
form. Then it goes on to minimum time with bounded jerks, and asks where the time goes. It uses the gradients, Jacobians and reverse-mode derivatives of the
{doc}`prerequisite lecture <p1_jacobians_and_hessians>`.

## 1. What broke

Each row was the next obstacle once the one above was out of the way. The times are
from one 16-core machine, and are a guide rather than a benchmark.

| What was measured | Before | After | What it took |
|---|---|---|---|
| deriving one arm's conditions | 400 s at 3 joints, never finished at 6 | a few seconds at 6 joints | a tape (Section 3) |
| kernels per set of limits ridden | one each, for 728 sets | one for all of them | solving at run time (Section 4) |
| compiling a long Jacobian | 159 s for one function | about 20 s | cutting it into pieces |
| the search for the structure | stalled on overlapping arcs | most of the motions surveyed (Section 6) | four fixes in the search, and correct second derivatives |
| the first guess in minimum time | no random draw landed in minutes | no draws at all | each joint alone, from a long time down (Section 7) |
| integrating the state transition matrix | 94% of a minimum-time solve | 54 s against 307 s | Jacobians only when asked for, at the accuracy needed (Section 7) |

It is worth being exact about what the slow part was. One call to the compiled dynamics
takes about 4 microseconds and one Jacobian about 25, so the kernels themselves were
never the problem. How many times they were called was.

## 2. A robot in a loop

In 1980 J. Y. S. Luh, M. W. Walker and R. P. C. Paul published an algorithm for a
robot's torques that was fast enough to run while the robot moved: the **recursive
Newton–Euler algorithm**. Velocities and accelerations are passed from the base
out to the hand, one link at a time, and then forces and moments are passed back from
the hand to the base. The work grows linearly with the number of joints, where the
Lagrangian formulations then in use grew far faster.

The package has it in `robot.joint_torques`, working on SymPy symbols. Each link is one
row of a table.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.robot import GRAVITY, SIX_JOINT_LINKS, joint_torques, link_frames
from auto_optimal_control.tape import Tape

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

q, w, a = (sp.symbols(f"{s}1:7", real=True) for s in "qwa")
tau = joint_torques(SIX_JOINT_LINKS, q, w, a, GRAVITY)
print("one link:", SIX_JOINT_LINKS[1])
print("operations in each torque, written out:", [sp.count_ops(e) for e in tau])

The first torque has about a hundred thousand operations. But the algorithm did not
do a hundred thousand operations to compute it. Written out, every intermediate result
is copied into everywhere it is used; the recursion computes it once.

## 3. Keep the program: the tape

A `Tape` records each distinct subexpression once, as a straight-line program, and
it applies the rules of differentiation to the program. This is the automatic
differentiation of the prerequisite lecture.

In [ ]:
tape = Tape()
roots = [tape.from_sympy(e) for e in tau]
print("steps for all six torques:", len(tape.reachable(roots)))

names = [s.name for s in (*q, *w, *a)]
start = time.perf_counter()
gradients = [tape.gradient(r, names) for r in roots]
print(f"all 108 first derivatives: {len(tape.reachable(sum(gradients, [])))} steps, "
      f"in {time.perf_counter() - start:.3f} s")

# Check against a difference of the formula as SymPy wrote it.
f = sp.lambdify((q, w, a), tau, "numpy")
rng = np.random.default_rng(0)
Q, W, A = rng.uniform(-1, 1, (3, 6))
values = dict(zip(names, np.concatenate([Q, W, A])))
step = 1e-6
difference = (np.array(f(Q + step * np.eye(6)[2], W, A), dtype=float)
              - np.array(f(Q - step * np.eye(6)[2], W, A), dtype=float)) / (2 * step)
print("d tau / d q3, tape vs difference:", np.abs(
    np.array(tape.evaluate([g[2] for g in gradients], values)) - difference).max())

**Nothing changes for the user.** You still write the limit as a SymPy expression. When
`derive` meets a path constraint longer than 200 operations, it records it on a tape and
puts a small opaque function, `Outlined`, in its place. SymPy can differentiate that
function: it asks the tape, and gets either another opaque function or zero. So
the rest of the derivation still sees short expressions. The two limits on one joint
share one *root*, because they are $\pm\tau_i$ minus a constant.

In [ ]:
problem = examples.six_joint_arm()
start = time.perf_counter()
pmp = derive(problem)
print(f"derive: {time.perf_counter() - start:.1f} s")
print("which root each limit uses:", pmp.riding.root_of)
print("and with what sign        :", pmp.riding.sign_of)
print("the first limit is now    :", str(pmp.path_constraints[0].expr)[:40], "...")
print("ways to ride limits at once:", len(pmp.path_constraints) + len(pmp.boundary_sets),
      "= 3^6 - 1: each joint at its upper limit, its lower one, or neither")

## 4. Solve when needed: one kernel for every set

Lecture 14 found the control that rides a limit by solving the riding conditions
symbolically, once per set of limits. Six limits at once would mean inverting a matrix
of mass-matrix entries in closed form, 728 times over.

But those equations are linear in the unknowns. For limits with
$\dot g_i = A_i(x)\,u + b_i(x)$ and a Hamiltonian quadratic in $u$, the controls and
multipliers solve

$$
\begin{bmatrix} R & A^\top \\ A & 0 \end{bmatrix}
\begin{bmatrix} u \\ \mu \end{bmatrix}
=
\begin{bmatrix} -\,\partial H/\partial u\,|_{u=0} \\ -b \end{bmatrix},
\qquad R = H_{uu},
$$

and C can solve that where it is needed, for whichever limits are active. The arc's
Jacobian, which the state transition matrix needs, comes from differentiating the same
system. Writing $y = (u, \mu)$ and calling these equations $r(z, y) = 0$,

$$
\frac{\partial y}{\partial z} = -K^{-1} \frac{\partial r}{\partial z},
$$

where $K$ is the matrix above, already factored. The second derivatives of the limits
enter only through one number, $\sum_k \mu_k \dot h_k$ over the roots $h_k$, whatever the
number of active limits. A single reverse pass gives its gradient, which is why the
Jacobian takes so few steps.

So `riding.py` generates one set of functions, and the arc is chosen by an argument. A
useful check, independent of how it was derived: along an arc, the limits it rides
should stop changing.

In [ ]:
start = time.perf_counter()
k = build_kernels(pmp)
print(f"compiling, or loading from the cache: {time.perf_counter() - start:.0f} s")

params = np.zeros(len(pmp.params))
params[0], params[-6:] = 1.0, 50.0             # T, and the torque limits
z = np.random.default_rng(1).uniform(-1, 1, k.N_Z)
step = 1e-6
for arc in (0, k.boundary_set([0, 3]), k.boundary_set([0, 2, 4, 6, 8, 10])):
    dz = k.boundary_rhs(arc, 0.3, z, params)
    rate = (k.constraint(0.3, z + step * dz, params) - k.constraint(0.3, z - step * dz, params)) / (2 * step)
    ridden = list(k.arc_constraints(arc))
    print(f"riding {len(ridden)}: largest rate among those ridden {np.abs(rate[ridden]).max():.1e}; "
          f"among the others {np.abs(np.delete(rate, ridden)).max():.1e}")
print("(riding one side of all six joints fixes every torque, so the other sides stop too)")

## 5. The arm solved

The same `Homotopy` call as Lecture 15, with no problem-specific code. The motion goes
from $q = 0$ to a pose with all six joints moved, in one second. Gravity sets a floor:
it holds joint 2 at 54 N·m just standing at the start pose, so a limit below that makes
the problem infeasible from $t = 0$, and `solve` says so immediately. Each limit below
is therefore set part of the way from the larger end-point torque to the peak of the
unconstrained motion, here 90% of the way, on all six joints at once.

In [ ]:
h = Homotopy(problem)
T, qf = 1.0, np.array([1.0, -1.0, 0.8, 1.0, -0.8, 1.0])
s = np.linspace(0, 1, 801)[:, None]
torque_of = lambda Q, W, A: np.array([np.broadcast_to(np.asarray(v, dtype=float), Q.shape[:1])
                                      for v in f(Q.T, W.T, A.T)]).T
unconstrained = torque_of(qf * (10 * s ** 3 - 15 * s ** 4 + 6 * s ** 5),
                          qf * (30 * s ** 2 - 60 * s ** 3 + 30 * s ** 4) / T,
                          qf * (60 * s - 180 * s ** 2 + 120 * s ** 3) / T ** 2)
peak = np.abs(unconstrained).max(axis=0)
ends = np.maximum(np.abs(unconstrained[0]), np.abs(unconstrained[-1]))
box = ([-50] * 18, [50] * 18)

loose = h.solve([T, *np.zeros(6), *qf, *(1.5 * peak)], box)
print(f"limits loose: J = {loose.objective:.9f}; six quintics give {360 * qf @ qf / T ** 5:.9f}")

limits = ends + 0.9 * (peak - ends)
start = time.perf_counter()
r = h.solve([T, *np.zeros(6), *qf, *limits], box)
print(f"all six limited: exact = {r.exact}, J = {r.objective:.6f}, in {time.perf_counter() - start:.1f} s")
print("the search's events:", [f"{kind} {name}" for kind, name in r.arcs][:6], "...")

impossible = h.solve([T, *np.zeros(6), *qf, *(0.5 * limits)], box)
print("with half those limits:", impossible.message[:80])

**An independent check.** As in Lecture 19, the jerks are replayed through plain SciPy
integration, and the torques recomputed from the formula SymPy wrote, without the
kernels or the tape.

In [ ]:
tr = r.shooting.trajectory(r.solution.v, n_points=20001)
t, u = tr["t"], tr["u"]
keep = np.concatenate([[True], np.diff(t) > 0])
jerk = interp1d(t[keep], u[keep], axis=0)
sol = solve_ivp(lambda ti, x: np.concatenate([x[6:], jerk(min(ti, t[-1]))]), (0, t[-1]), np.zeros(18),
                rtol=1e-10, atol=1e-12, max_step=1e-3, t_eval=np.linspace(0, t[-1], 2001))
torque = torque_of(sol.y[:6].T, sol.y[6:12].T, sol.y[12:].T)
replayed = float(np.trapezoid(0.5 * (u[keep] ** 2).sum(axis=1), t[keep]))
print(f"cost: {r.objective:.9f} vs replay {replayed:.9f}")
print(f"final state error: {np.abs(sol.y[:, -1] - np.concatenate([qf, np.zeros(12)])).max():.1e}")
print(f"largest torque over its limit: {(np.abs(torque) - limits).max():.1e}")

fig, axes = plt.subplots(2, 3, figsize=(11.5, 5.2), sharex=True, constrained_layout=True)
for i, ax in enumerate(axes.flat):
    ax.plot(sol.t, torque[:, i], color=BLUE, lw=2)
    ax.plot(s[:, 0] * T, unconstrained[:, i], color=GRAY, lw=1, ls="--")
    for sign in (1, -1):
        ax.axhline(sign * limits[i], color=ORANGE, lw=1)
    ax.set_title(f"joint {i + 1}")
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].plot([], [], color=BLUE, lw=2, label="the solution")
axes[0, 0].plot([], [], color=GRAY, lw=1, ls="--", label="without limits")
axes[0, 0].plot([], [], color=ORANGE, lw=1, label="the limit")
axes[0, 0].legend(frameon=False, fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("t")
for ax in axes[:, 0]:
    ax.set_ylabel("torque")
plt.show()

*Each joint's torque. Without limits, each joint's largest torque would pass its limit; the solution rides the limits there, with arcs overlapping. Every joint's motion changes a little because of the others' limits, which couple through the mass matrix. This is a stress test: a real arm's last joint does not have a limit near 1 mN·m.*

In [ ]:
from IPython.display import HTML
from matplotlib import animation

plt.rcParams["animation.embed_limit"] = 30
n_frames = 60
idx = np.linspace(0, len(sol.t) - 1, n_frames).astype(int)
riding = np.abs(np.abs(torque) - limits) < 1e-4
points = []
for i in idx:
    frames = link_frames(SIX_JOINT_LINKS, sol.y[:6, i])
    points.append(np.vstack([np.zeros(3), [p for _, p in frames]]))
points = np.array(points)
fig = plt.figure(figsize=(5.2, 4.6), dpi=72)
ax = fig.add_subplot(projection="3d")
ax.plot(*points[:, -1].T, color="k", lw=0.7, ls=":")
line, = ax.plot([], [], [], "-o", color=BLUE, lw=4, ms=5)
hot, = ax.plot([], [], [], "o", color=ORANGE, ms=10)
label = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=9)
span = 0.85
ax.set(xlim=(-span, span), ylim=(-span, span), zlim=(-0.3, 1.0), xticks=[], yticks=[], zticks=[])


def draw(n):
    p = points[n]
    line.set_data(p[:, 0], p[:, 1])
    line.set_3d_properties(p[:, 2])
    on = riding[idx[n]]
    hot.set_data(p[:-1, 0][on], p[:-1, 1][on])
    hot.set_3d_properties(p[:-1, 2][on])
    label.set_text(f"t = {sol.t[idx[n]]:.2f}   joints at a limit: {list(np.flatnonzero(on) + 1)}")
    return line, hot, label


anim = animation.FuncAnimation(fig, draw, frames=n_frames, interval=90, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

*The arm, moving: press play. A joint turns orange while it rides a torque limit. The dotted line is the path of the hand.*

## 6. How often does it work?

One worked example says little about a search. So the solve was run on random motions,
each with a random goal and duration, $T \in [0.8, 1.5]$ and $q_f \in [-1.5, 1.5]^6$, at
four limit tightnesses, with a time budget of 300 seconds.

| Limits tightened | Motions solved exactly | Median time when solved | 90th percentile |
|---|---|---|---|
| joints 1–3 (the arm), 95% to 70% | 40 of 48 (83%) | 2.5 s | 7 s |
| all six (also the wrist, at 80%) | 32 of 48 (67%) | 13 s | 45 s |

The other half of a search's quality is what it does on failure. A failure here is
reported as `exact = False`, never as a wrong answer, because a result is only accepted
when it passes the checks of Lecture 15.

The search needed four repairs that had nothing to do with the size of the arithmetic.
All four only appear when many arcs overlap:

- a violation that spanned the junction of another arc was reported as two violations,
  so the search tried to enter one limit twice;
- a tiny violation at a tangential junction, at the level of the path tolerance, sent
  the search round in a circle, re-activating the arc it already had;
- a new arc born inside another must be guessed with its events in the right order, and
  the solve only converges from the right one. Nested is the commonest;
- merging two arcs of one limit removed any arc of another limit that lay between them.

A fourth problem was a wrong derivative in the new code, of the kind the prerequisite
lecture warns about. It was found by comparing the Jacobian with finite differences.

## 7. Minimum time, with bounded jerks

This is Lecture 19's problem at full scale: $\min t_f$ with $|j_i| \le j_{\max}$ and
$|\tau_i| \le \tau_{\max,i}$. The statement is degenerate, since only the slowest joint sets
the time, so the default policy adds the small jerk cost of Lecture 19. Three new things
were needed, and only one of them is about bounds.

**Riding with bounds.** The system of Section 4 has no inequality in it. With bounds it
becomes a small quadratic program, strictly convex, so its solution is unique and a function
of $z$. It is found by holding controls at their bounds and adding those rows to the same
linear system, until the controls are in bounds and the bounds' multipliers have the right
signs. A held control does not move with $z$, so the rule for $\partial y/\partial z$ does
not change at all.

**A first guess without guessing.** Shooting needs initial costates. Random draws must land
all six joints at once, and a nearly bang-bang control gives Newton almost nothing to work
with: they found nothing in minutes. But free arcs do not couple the joints, so each is a
small problem, and it has an easy end. Given plenty of time a joint's bounds are idle and its
cost is nearly quadratic, so Newton needs no guess there. From that end its costates follow
continuation as its time shrinks, until $H$ reaches zero: its own minimum time. The joints
then share the cost of time, so the common $t_f$ sits where their parts of $H$ add up to zero.

**Where the time goes.** A profile of the first working solve answered the question of
whether the barrier was the costates or the integration: 288 of its 307 seconds were free
arcs integrated with the state transition matrix, and only 17 nonlinear solves were made.

| Change | Why it is safe | Effect |
|---|---|---|
| compute a Jacobian only when the solver asks for one | MINPACK asks for far more residuals than Jacobians, and a free arc costs about 2 ms without $\Phi$ and 76 ms with it | 307 s to 108 s |
| a looser tolerance on $\Phi$ along the path | the path is followed to $10^{-6}$, and a rough Jacobian steers Newton as well | fewer steps |
| skip zeros in $A\Phi$ | on a free arc $A$ is mostly zeros | with the line above, half the cost per step |
| all of them | the final exact solve keeps full accuracy | **307 s to 54 s, the same $t_f$ to all digits** |

The last column of that table had a trap in it. Applied to the final solve as well, faster
residuals stopped at $2.7 \times 10^{-10}$, above the tolerance, because each saturation
corner carries the error of the step that found it, and these residuals take longer steps. So
the search uses them, and the last solve does not.

In [ ]:
limited = Homotopy(examples.six_joint_arm_min_time())
J_max = 30.0
goal = np.array([1.0, -0.9, 0.8, 1.1, -0.7, 0.6])
base = [*np.zeros(6), *goal, *([J_max] * 6)]
box, times = ([-0.5] * 18, [0.5] * 18), (0.3, 5.0)

start = time.perf_counter()
free = limited.solve([*base, *([1e4] * 6)], box, times)
print(f"limits loose: t_f = {free.solution.tf:.6f} in {time.perf_counter() - start:.0f} s, with no random draws "
      f"(tries = {free.tries}); joint 4 alone would take {(32 * 1.1 / J_max) ** (1 / 3):.6f}")

path = free.shooting.trajectory(free.solution.v, n_points=4001)
reached = torque_of(path["z"][:, :6], path["z"][:, 6:12], path["z"][:, 12:18])
peaks, floors = np.abs(reached).max(axis=0), 1.001 * np.maximum(np.abs(reached[0]), np.abs(reached[-1]))
bounds = floors + 0.9 * (peaks - floors).clip(0)

start = time.perf_counter()
fast = limited.solve([*base, *bounds], box, times)
print(f"all six limited: exact = {fast.exact}, t_f = {fast.solution.tf:.6f}, in {time.perf_counter() - start:.0f} s")
print("the search's events:", [f"{kind} {name}" for kind, name in fast.arcs])

tr = fast.shooting.trajectory(fast.solution.v, n_points=40001)
t, u = tr["t"], tr["u"]
keep = np.concatenate([[True], np.diff(t) > 0])
jerk = interp1d(t[keep], u[keep], axis=0)
sol = solve_ivp(lambda ti, x: np.concatenate([x[6:], jerk(min(ti, t[-1]))]), (0, t[-1]), np.zeros(18),
                rtol=1e-10, atol=1e-12, max_step=1e-3, t_eval=np.linspace(0, t[-1], 4001))
torque = torque_of(sol.y[:6].T, sol.y[6:12].T, sol.y[12:].T)
replayed = t[-1] + 0.01 * float(np.trapezoid(0.5 * ((u[keep] / J_max) ** 2).sum(axis=1), t[keep]))
print(f"cost: {fast.objective:.9f} vs replay {replayed:.9f}")
print(f"final state error: {np.abs(sol.y[:, -1] - np.concatenate([goal, np.zeros(12)])).max():.1e}; "
      f"largest torque over its limit: {(np.abs(torque) - bounds).max():.1e}; largest |j|: {np.abs(u).max():.4f}")

fig, axes = plt.subplots(6, 2, figsize=(10.5, 7.2), sharex=True, constrained_layout=True)
for i in range(6):
    axes[i, 0].plot(t[keep], u[keep, i] / J_max, color=GRAY, lw=1.5)
    axes[i, 1].plot(sol.t, torque[:, i] / bounds[i], color=BLUE, lw=1.8)
    axes[i, 1].axhline(1, color=ORANGE, lw=1)
    axes[i, 1].axhline(-1, color=ORANGE, lw=1)
    axes[i, 0].axhline(1, color="k", lw=0.5)
    axes[i, 0].axhline(-1, color="k", lw=0.5)
    axes[i, 0].set_ylabel(f"joint {i + 1}")
    for ax in axes[i]:
        ax.set_ylim(-1.25, 1.25)
        ax.set_yticks([-1, 1])
        ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].set_title("jerk / its limit")
axes[0, 1].set_title("torque / its limit")
axes[5, 0].set_xlabel("t")
axes[5, 1].set_xlabel("t")
plt.show()

*Six joints in minimum time. Joint 4 has the longest move, so it sets the time and is nearly bang-bang. The others have time to spare, and their jerks are smooth. Where a torque limit is ridden, a jerk leaves its bound or a kink appears in the others.*

These limits barely cost any time, because the slowest joint's limit does not bind. On ten
random goals, $q_f \in [-1.5, 1.5]^6$ with $j_{\max} = 30$ and limits tightened on every joint
together, the survey found (later rows reuse the first guess, so their times are the search's):

| Limits at | Motions solved exactly | Median time when solved | Largest rise in $t_f$ |
|---|---|---|---|
| no limits active | 10 of 10 | 38 s (the first guess) | none |
| 95% | 8 of 10 | 42 s | 0.0006 s |
| 90% | 5 of 10 | 52 s | 0.0001 s |
| 80% | 3 of 10 | 57 s | 0.012 s |
| 70% | 2 of 10 | 239 s | 0.040 s |

So the method does reach cases where torque slows the arm, but this is the weakest part of the
package. **The open problem is here.** When the limits get tighter on the joint that sets the
time, the search often stalls as a new limit comes into play, typically the opposite limit
on a joint already riding one. A bang-bang joint's torque peaks at a sharp corner, so one possible
cause is that its arrival at the limit is not tangential, but this has not been confirmed.
Failures end as `exact = False`, not as wrong answers.

## 8. A limit on the derivative of a control

Every limit so far has been on a control or on something the control integrates into: an
*antiderivative*. A natural question runs the other way. Suppose acceleration is what you
command, and its *rate* is limited: $v' = a$ with $|a'| \le J$. The limit is on a derivative
of the control. Why not just add it to the problem?

**Why the minimum principle cannot take it.** Lecture 4 proved the principle with needle
variations: replace the control on a tiny interval by *any* admissible value, measure the
change in the cost, and conclude that the best control minimizes $H$ pointwise. Two things in
that argument need the set of allowed values $U$ to be a fixed set, or at least a function of
$t$ and $x$:

- the needle must be admissible. With a rate limit it never is, because a jump in $a$ has
  infinite rate;
- $H$ must decide the control from $(t, x, \lambda)$ alone. With a rate limit what is allowed
  now depends on the value of the control a moment ago: on its history, which is not in
  $(t, x)$.

So $|a'| \le J$ is not a constraint of the form $g(t, x, u) \le 0$, and no choice of $H$ makes
pointwise minimization respect it. The obvious workaround, solving without the rate limit
and then passing the answer through a rate limiter, fails in a way we can measure. Here is one
axis moving a distance $D$ in minimum time with $|a| \le A$:

In [ ]:
D_axis, A_axis, J_axis = 1.0, 2.0, 8.0
T_bang = 2 * np.sqrt(D_axis / A_axis)         # acceleration control: +A, then -A

times = np.linspace(0, T_bang, 200001)
step = times[1] - times[0]
command = np.where(times < T_bang / 2, A_axis, -A_axis)
limited = np.zeros_like(times)                # the same command through a rate limiter
for i in range(1, len(times)):
    limited[i] = limited[i - 1] + np.clip(command[i] - limited[i - 1], -J_axis * step, J_axis * step)
speed = np.concatenate([[0.0], np.cumsum(0.5 * (limited[1:] + limited[:-1]) * step)])
place = np.concatenate([[0.0], np.cumsum(0.5 * (speed[1:] + speed[:-1]) * step)])
print(f"bang-bang: t_f = {T_bang:.6f}")
print(f"through a rate limiter, at t_f: position {place[-1]:.4f} (want {D_axis}), "
      f"speed {speed[-1]:.4f} (want 0), acceleration {limited[-1]:.4f} (want 0)")

It misses on all three counts, and adjusting afterwards gives a feasible motion, not the
fastest one. The rate limit has to be part of the optimization.

**The fix is one line of modelling.** Make the thing whose rate is limited a *state*, and its
rate the new control: $a' = j$ with $|j| \le J$. Now the limit is an ordinary box, and
the needles are in $j$. That is the jerk-controlled problem this course has been solving
since Lecture 18, so "acceleration control with a jerk limit" is not a new problem. The two
descriptions are the same.

In [ ]:
axis = Homotopy(examples.jerk_limited_move())
speed_limit = 100.0                           # far out of reach
result = axis.solve([J_axis, A_axis, speed_limit, D_axis], ([-3] * 3, [3] * 3), (0.5, 30.0))
closed = examples.jerk_limited_min_time(J_axis, A_axis, speed_limit, D_axis)[0]
print(f"with a as a state: t_f = {result.solution.tf:.9f}; closed form {closed:.9f}; "
      f"{closed / T_bang - 1:.1%} slower than the bang-bang motion")
print("arcs found:", [name for _, name in result.arcs])

curve = result.shooting.trajectory(result.solution.v, n_points=4001)
fig, ax = plt.subplots(figsize=(8.0, 3.8))
ax.plot(times, command, color=GRAY, lw=1.5, label="acceleration control: jumps")
ax.plot(times, limited, color=ORANGE, lw=2, ls="--", label="the same, through a rate limiter: misses the goal")
ax.plot(curve["t"], curve["z"][:, 2], color=BLUE, lw=2.5, label="acceleration as a state: the optimum")
ax.set(xlabel="t", ylabel="acceleration")
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

*Three accelerations for one move. The rate-limited version lags the command and is still at full deceleration when time runs out. The true optimum rides $+A$, turns at the jerk limit, and arrives at rest, taking longer.*

**What it costs: every other limit moves up one order.** Making $a$ a state puts one more
integration between the control and everything else, so each limit needs one more
differentiation before a control appears. `constraint_derivatives` counts them:

In [ ]:
import warnings

from auto_optimal_control.pontryagin import constraint_derivatives
from auto_optimal_control.problem import OptimalControlProblem

t_, x_, v_, a_, j_ = sp.symbols("t x v a j", real=True)
L_ = sp.Symbol("L", positive=True)
by_acceleration = OptimalControlProblem(name="a", time=t_, states=[x_, v_], controls=[a_], params=[L_],
                                        dynamics=[v_, a_], running_cost=1, initial_state=[0, 0])
by_jerk = OptimalControlProblem(name="j", time=t_, states=[x_, v_, a_], controls=[j_], params=[L_],
                                dynamics=[v_, a_, j_], running_cost=1, initial_state=[0, 0, 0])
for name, g in (("speed", v_ - L_), ("position", x_ - L_)):
    print(f"{name:9s} limit: order {len(constraint_derivatives(by_acceleration, g)) - 1} with acceleration "
          f"control, order {len(constraint_derivatives(by_jerk, g)) - 1} with a jerk limit")
print("accel     limit: on the control with acceleration control, "
      f"order {len(constraint_derivatives(by_jerk, a_ - L_)) - 1} with a jerk limit")

| Limit | Acceleration control, no jerk limit | With a jerk limit |
|---|---|---|
| acceleration | a box on the control | order 1 |
| torque, $\tau(q, w, a)$ | contains the control: order 0, with no costate jumps | order 1, with the junctions of Lecture 14 |
| speed | order 1 | order 2 |
| position | order 2 | order 3 |

The last row is a real loss. Lecture 13 explained that riding a limit of order 3 generally
chatters, with infinitely many touch points, so a position limit under a jerk limit has no
simple exact solution. That comes from the jerk limit, not from how the problem is written:
any formulation of the same problem has to face it.

**Two ways of posing it change the problem itself.**

- *A cost on acceleration, such as $\int a^2$.* With $a$ a state the cost no longer contains the
  control, so $H$ is linear in $j$: bang-bang at the limit, with singular arcs (Lecture 12) in
  between. Several joints would be singular at once, which the package does not handle, and
  the arcs solved at run time need a cost strictly convex in the controls. A small
  $w \int j^2$, as in Section 7, removes both obstacles.
- *A rate limit on torque.* If torque is the state, $w' = M(q)^{-1}(\tau - c)$. The joints are
  no longer decoupled on free arcs and the dynamics are no longer a simple chain, so the
  first guess of Section 7 no longer applies and much of the speed is gone.

The end conditions also need thought. A state needs an initial value, so the acceleration at
$t = 0$ must be known. If the final one is free, transversality gives $\lambda_a(t_f) = 0$
in place of a fixed value.

## 9. What this does not do yet

- **Torque limits that set the time, reliably.** Section 7's survey shows how far this goes.
- **Limits of order 2 at this size,** such as a velocity limit on a joint of the six-joint
  arm. Small problems still get them from the symbolic route.
- **A better search for many short arcs.** That is where the remaining failures in Section 6
  are, and where the remaining time goes.
- **Global optimality.** As everywhere in this course, these are extremals: they satisfy
  the necessary conditions. A six-joint arm can have several.

Learned warm starts remain a possible last step. What this lecture shows is that they
were not needed to reach six joints. With the waste taken out, the first guess takes about
half of a solve and the search the rest, which is what a warm start would target.